# Linear Regression Model Building Phase

### Load dataset and check first few rows

In [16]:
from sklearn.datasets import fetch_openml
import pandas as pd

pd.set_option("display.max_columns", None)
df = pd.read_csv("../data/cleaned/listings_features.csv")

print(df.shape)
df.head(10)

(8760, 25)


,host_is_superhost,neighbourhood_cleansed,accommodates,bathrooms_text,bedrooms,price_quote_price_per_night,minimum_nights,review_scores_rating,calculated_host_listings_count,has_license,distance_band,amenity_count,has_pool,has_hot_tub,has_gym,has_game_room,has_fire_pit,has_ev_charger,pets_allowed,has_city_skyline_view,has_free_parking,room_type_grouped,property_group,log_price,price_outlier_flag
0,t,District 25,2,1 private bath,1.0,80.00,1.0,4.94,1,0,5-10 km,27,0,0,0,0,0,0,0,0,1,Room,House,4.382027,0
1,t,District 18,2,1 bath,2.0,132.00,2.0,4.92,1,0,3-5 km,42,0,0,0,0,0,0,0,0,0,Entire place,Apartment/Condo,4.882802,0
2,t,District 19,2,1 bath,1.0,123.50,1.0,4.90,7,0,0-3 km,43,0,0,0,0,0,0,0,0,1,Entire place,Guest Unit,4.816241,0
3,t,District 6,2,1 private bath,1.0,114.00,1.0,4.86,1,0,0-3 km,29,0,0,0,0,0,0,0,0,1,Room,House,4.736198,0
4,t,District 19,2,1 bath,1.0,128.00,1.0,4.95,7,0,0-3 km,42,0,0,0,0,0,0,0,1,1,Entire place,Guest Unit,4.852030,0
5,t,District 19,2,1 bath,1.0,128.00,1.0,4.96,7,0,0-3 km,40,0,0,0,0,0,0,0,1,1,Entire place,Guest Unit,4.852030,0
6,t,District 19,4,2.5 baths,2.0,381.00,1.0,4.89,7,0,0-3 km,61,0,1,0,0,1,0,0,0,1,Entire place,House,5.942799,0
7,f,District 14,4,1 bath,1.0,NaN,2.0,4.70,1,0,10+ km,7,0,0,0,0,0,0,0,0,1,Room,House,NaN,0
8,t,District 19,2,1.5 baths,1.0,203.50,1.0,4.97,7,0,0-3 km,51,0,0,0,0,0,0,0,1,1,Entire place,Apartment/Condo,5.315666,0
9,f,District 6,2,1 bath,1.0,170.67,3.0,4.93,2,0,3-5 km,45,0,0,0,0,0,0,0,0,1,Entire place,Guest Unit,5.139732,0


### Drop missing values in target variable

In [17]:
df = df.dropna(subset=['log_price'])

# Ensure that there are no more missing target values
df['log_price'].isnull().sum()

np.int64(0)

### Convert bathroom text to numbers

Currently, the bathroom_text variable contains listings like "1.5 baths" and "shared bath" which would create an excess of dummy variables. To fix this, converting these values to a number first will reduce the amount of dummies created

In [18]:
df = df.copy()
txt = df['bathrooms_text'].str.lower()

# Number of baths: first number in the text (e.g. "1.5 baths" -> 1.5)
df['bathrooms'] = txt.str.extract(r'(\d+\.?\d*)')[0].astype(float)

# "Half-bath" / "Shared half-bath" have no number, so count them as 0.5
df.loc[txt.str.contains('half', na=False) & df['bathrooms'].isnull(), 'bathrooms'] = 0.5

# 1 if the bath is shared, 0 otherwise
df['shared_bath'] = txt.str.contains('shared', na=False).astype(int)

# The original text column is now replaced
df = df.drop(columns=['bathrooms_text'])

# Check the result: any remaining blanks (imputed later, after the split) and the value counts
print("Missing bathrooms:", df['bathrooms'].isnull().sum())
print(df['bathrooms'].value_counts().sort_index())
print(df['shared_bath'].value_counts())

Missing bathrooms: 12
bathrooms
0.0        9
0.5        3
1.0     3187
1.5      179
2.0     1436
2.5      540
3.0      607
3.5      695
4.0      740
4.5      242
5.0       80
5.5       36
6.0       79
6.5       32
7.0       67
7.5       15
8.0      106
8.5        9
9.0       41
9.5        8
10.0      13
10.5       5
11.0       1
12.0      15
12.5       3
13.0       5
13.5       2
14.0       4
15.0       3
16.0       6
16.5       2
17.0       1
17.5       1
18.0       2
19.0       1
20.0       2
21.0       1
23.0       1
24.0       2
40.5       1
47.0       1
50.0       2
Name: count, dtype: int64
shared_bath
0    8129
1      68
Name: count, dtype: int64


### Train/Test Split

In [19]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size = 0.25, random_state = 1234)

### Set target variable and drop-non predictors

In [20]:
import statsmodels.api as sm

predictors = train.drop(columns=['log_price','price_quote_price_per_night','price_outlier_flag'])

X = predictors.copy()
y = train['log_price']

### Impute missing values (Continous Variables)

In [21]:
num_cols = X.select_dtypes(include='number').columns

# Store the train medians so the same values can fill missing data in the test set later
medians = {}

for col in num_cols:
    if X[col].isnull().any():
        # Create missing flag column
        X[f'{col}_was_missing'] = X[col].isnull().astype(int)
        
        # Impute with median
        median = X[col].median()
        medians[col] = median
        X[col] = X[col].fillna(median)

print(medians)

{'bedrooms': np.float64(2.0), 'minimum_nights': np.float64(1.0), 'review_scores_rating': np.float64(4.91), 'bathrooms': np.float64(2.0)}


### Create dummy variables and ensure no missing values remain

In [22]:
X = pd.get_dummies(X, drop_first=True)
X = X.astype(float)

print(X.shape)
print("Missing values left:", X.isnull().sum().sum())

(6147, 66)
Missing values left: 0
